# 02 · Detection, P1/P2 tracking and pose

Phase 2 of the pipeline. Reads a 60 fps video and writes to `data/runs/<video_stem>/`:

| File | What |
|---|---|
| `frames.json` | fps, size, frame count, HUD mask |
| `detections.npz` | every RTMDet person box (with a colour histogram), plus the chosen P1/P2 boxes per frame |
| `keypoints.npz` | `(T, 2, 17, 3)` RTMPose COCO-17 keypoints: x, y in pixels and a score; NaN where a fighter wasn't found |
| `annotated.mp4` | the video with fighter boxes and skeletons |

Every frame is processed; frame `t` in every output is frame `t` (0-based) of the video.

**How to run:** pick the `.venv` Python as the kernel, set the parameters in the next cell, then *Restart & Run All*.
Needs the pretrained weights (`python scripts/download_weights.py`).

In [ ]:
# Parameters
CONFIG = "configs/default.yaml"                               # repo-relative
VIDEO = "data/raw/goto-ken_vs_haitani-chunli_0330-0350.mp4"  # repo-relative or absolute
OVERRIDES = ["device=mps"]  # Mac: mps · NVIDIA: cuda · otherwise cpu (detection then takes ~0.4 s/frame)
RENDER_VIDEO = True         # write annotated.mp4

In [ ]:
import copy
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

from sf6mr import frames as fr
from sf6mr import runs, track, visualize
from sf6mr.config import load_config, resolve_path

warnings.filterwarnings("ignore", category=UserWarning)

cfg = load_config(resolve_path(CONFIG), OVERRIDES)
device = cfg["device"]
video = resolve_path(VIDEO)
out_dir = runs.run_dir(cfg, video)

info = fr.read_video_info(video, expected_fps=cfg["video"]["fps"])
T = info.frame_count
hud = cfg.get("hud_mask") or {}
hud_px = fr.hud_rects_px(hud.get("regions", []), info.width, info.height) if hud.get("enabled") else np.zeros((0, 4), int)
runs.save_frames_json(out_dir / "frames.json", {
    **info.to_dict(),
    "hud_mask": {"enabled": bool(hud.get("enabled")), "regions": hud.get("regions", []), "regions_px": hud_px.tolist()},
})
print(f"{video.name}: {info.width}x{info.height} @ {info.fps:g} fps, {T} frames ({T / info.fps:.1f} s)")
print(f"device: {device} · outputs: {out_dir}")

## Detection

RTMDet-m (person only). Spectators on the stage are detected too; the tracker below picks the fighters.
On MPS the network runs on the GPU and only the final NMS runs on the CPU, because MMCV's NMS has no MPS kernel
(same boxes as CPU, ~25× faster).

In [ ]:
from mmdet.apis import init_detector
from mmdet.utils import register_all_modules as register_mmdet
from mmengine.config import Config
from mmengine.dataset import Compose, pseudo_collate

register_mmdet()
det_cfg = cfg["detection"]
detector = init_detector(
    Config.fromfile(resolve_path(det_cfg["config"])),
    checkpoint=str(resolve_path(det_cfg["checkpoint"])),
    device=device,
)
_pipeline = copy.deepcopy(detector.cfg.test_dataloader.dataset.pipeline)
_pipeline[0] = dict(type="mmdet.LoadImageFromNDArray")
det_pipeline = Compose(_pipeline)


@torch.no_grad()
def detect(batch):
    """RTMDet on a list of BGR frames -> one (n, 5) array [x0, y0, x1, y1, score] per frame."""
    data = pseudo_collate([det_pipeline(dict(img=f, img_id=i)) for i, f in enumerate(batch)])
    data = detector.data_preprocessor(data, False)
    outs = detector.bbox_head(detector.extract_feat(data["inputs"]))
    if device.startswith("mps"):  # NMS on the CPU
        outs = tuple([o.float().cpu() for o in level] for level in outs)
    metas = [s.metainfo for s in data["data_samples"]]
    results = detector.bbox_head.predict_by_feat(*outs, batch_img_metas=metas, rescale=True)
    return [np.hstack([r.bboxes.cpu().numpy(), r.scores.cpu().numpy()[:, None]]) for r in results]

In [ ]:
det_frame, det_boxes, det_hist = [], [], []


def run_batch(frames_, indices):
    masked = [fr.apply_hud_mask(f, hud_px) for f in frames_]
    for t, frame, dets in zip(indices, frames_, detect(masked)):
        for d in dets[dets[:, 4] >= det_cfg["save_score"]]:
            det_frame.append(t)
            det_boxes.append(d)
            det_hist.append(track.color_hist(frame, d))


t0 = time.perf_counter()
batch, indices = [], []
for t, frame in tqdm(fr.iter_frames(video), total=T, desc="detect"):
    batch.append(frame)
    indices.append(t)
    if len(batch) == det_cfg["batch_size"]:
        run_batch(batch, indices)
        batch, indices = [], []
if batch:
    run_batch(batch, indices)

det_frame = np.asarray(det_frame, np.int32)
det_boxes = np.asarray(det_boxes, np.float32).reshape(-1, 5)
det_hist = np.asarray(det_hist, np.float32).reshape(-1, track.HIST_DIM)
print(f"{len(det_frame)} boxes ≥ {det_cfg['save_score']} · {(time.perf_counter() - t0) / T * 1000:.0f} ms/frame")

## P1 / P2

`sf6mr.track.assign_fighters` picks the two fighters in each frame and keeps their identity through side switches,
using the ground line (spectators stand behind the fence), frame-to-frame continuity and a colour template per fighter.
P1 is whoever is on the left when both are first found. When it isn't sure, a fighter is left **missing** (NaN) rather
than guessed: typically during Super cinematics, heavy overlaps and fast knockdowns.

In [ ]:
params = track.TrackerParams.from_config(cfg.get("tracker"))
fighters, idx = track.assign_fighters(
    runs.split_by_frame(det_frame, det_boxes, T),
    info.height,
    params,
    features=runs.split_by_frame(det_frame, det_hist, T),
)
det_index = runs.to_flat_index(det_frame, idx)
runs.save_detections(out_dir / "detections.npz", det_frame, det_boxes, det_hist, fighters, det_index, params)

found = ~np.isnan(fighters[..., 0])
print(f"found: P1 {found[:, 0].mean():.0%} · P2 {found[:, 1].mean():.0%} · both {found.all(1).mean():.0%} of frames")


def gaps(mask, min_len=10):
    m = np.r_[0, (~mask).astype(int), 0]
    starts, ends = np.flatnonzero(np.diff(m) == 1), np.flatnonzero(np.diff(m) == -1)
    return [(int(a), int(b - 1)) for a, b in zip(starts, ends) if b - a >= min_len]


for p in range(2):
    print(f"P{p + 1} missing for ≥10 frames:", gaps(found[:, p]) or "none")

In [ ]:
cx = (fighters[..., 0] + fighters[..., 2]) / 2
fig, ax = plt.subplots(figsize=(12, 3.5))
for p, color in enumerate(["#d6337f", "#1f7fe0"]):
    ax.plot(np.arange(T), cx[:, p], ".", ms=2, color=color, label=f"P{p + 1}")
ax.set(xlim=(0, T), ylim=(0, info.width), xlabel="frame", ylabel="box center x (px)",
       title="Where each fighter is (gaps = not found; lines crossing = side switch)")
ax.legend(loc="upper right", markerscale=6)
plt.show()

## Pose

RTMPose-m (COCO-17, 256×192), top-down on each fighter's box. Scores are SimCC confidences: higher is better,
but they are not probabilities and can exceed 1.

In [ ]:
from mmpose.apis import inference_topdown, init_model
from mmpose.utils import register_all_modules as register_mmpose

register_mmpose()
pose_cfg = cfg["pose"]
pose_model = init_model(
    str(resolve_path(pose_cfg["config"])), str(resolve_path(pose_cfg["checkpoint"])), device=device
)

keypoints = np.full((T, 2, 17, 3), np.nan, np.float32)
t0 = time.perf_counter()
for t, frame in tqdm(fr.iter_frames(video), total=T, desc="pose"):
    players = [p for p in range(2) if found[t, p]]
    if not players:
        continue
    results = inference_topdown(pose_model, frame, fighters[t, players, :4], bbox_format="xyxy")
    for p, r in zip(players, results):
        keypoints[t, p, :, :2] = r.pred_instances.keypoints[0]
        keypoints[t, p, :, 2] = r.pred_instances.keypoint_scores[0]
runs.save_keypoints(out_dir / "keypoints.npz", keypoints)
print(f"{(time.perf_counter() - t0) / T * 1000:.0f} ms/frame")

In [ ]:
JOINTS = ["nose", "L eye", "R eye", "L ear", "R ear", "L shoulder", "R shoulder", "L elbow", "R elbow",
          "L wrist", "R wrist", "L hip", "R hip", "L knee", "R knee", "L ankle", "R ankle"]
scores = keypoints[..., 2]  # (T, 2, 17)
fig, ax = plt.subplots(figsize=(12, 3.5))
w = 0.4
for p, color in enumerate(["#d6337f", "#1f7fe0"]):
    ax.bar(np.arange(17) + (p - 0.5) * w, np.nanmedian(scores[:, p], axis=0), w, color=color, label=f"P{p + 1}")
ax.axhline(cfg["render"]["min_joint_score"], color="gray", lw=1, ls="--", label="drawing threshold")
ax.set_xticks(np.arange(17), JOINTS, rotation=45, ha="right")
ax.set(ylabel="median score", title="RTMPose confidence per joint (frames where the fighter was found)")
ax.legend()
plt.tight_layout()
plt.show()

## Annotated video

Boxes and skeletons in P1 pink / P2 blue (left-side limbs lighter), frame number bottom-left.

In [ ]:
if RENDER_VIDEO:
    rc = cfg["render"]
    with visualize.VideoWriter(out_dir / "annotated.mp4", info.fps, (info.width, info.height), crf=rc["crf"]) as writer:
        for t, frame in tqdm(fr.iter_frames(video), total=T, desc="render"):
            writer.write(visualize.draw_frame(frame, fighters[t], keypoints[t], t, rc["min_joint_score"]))
    print("wrote", out_dir / "annotated.mp4")

In [ ]:
preview = set(np.linspace(0, T - 1, 6).astype(int).tolist())
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
shown = 0
for t, frame in fr.iter_frames(video):
    if t in preview:
        img = visualize.draw_frame(frame, fighters[t], keypoints[t], t, cfg["render"]["min_joint_score"])
        axes.flat[shown].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes.flat[shown].axis("off")
        shown += 1
plt.tight_layout()
plt.show()